# Module 3: KPI Engineering - Military Metrics

This notebook demonstrates how to generate KPIs and enrich data with metadata.

## Overview
- Load cleaned data
- Calculate 5 strategic KPIs
- Add regional classification (5 regions)
- Add alliance metadata (NATO, BRICS, ASEAN, EU, SCO)
- Create wide format (analysis)
- Create long format (Tableau-optimized)
- Validate calculations

## 1. Setup and Load Data

In [ ]:
import sys
sys.path.append('../')

from generate_kpis import KPIGenerator
import pandas as pd
import numpy as np

# Initialize KPI generator
generator = KPIGenerator(
    config_file='kpi_definitions.json',
    log_file='kpi_generation_log.txt'
)

print("✅ KPI Generator initialized")

## 2. Load Cleaned Data

In [ ]:
# Load cleaned data from Milestone 1
df = pd.read_csv('../Module_2_Data_Cleaning/data/processed/military_cleaned.csv')

print(f"📊 Input Data Shape: {df.shape}")
print(f"\n📋 Sample Data:")
print(df.head())

## 3. Calculate 5 KPIs

In [ ]:
# Calculate KPIs
print("📊 Calculating KPIs...\n")

df_kpi = generator.calculate_all_kpis(df)

print(f"✅ KPI Calculation Complete")
print(f"\nKPIs Generated:")
kpi_list = [
    "1. Power Index Rank Gap",
    "2. Assets per Capita",
    "3. Budget-to-GDP Ratio",
    "4. Personnel Density",
    "5. Equipment Density"
]
for kpi in kpi_list:
    print(f"   ✓ {kpi}")

## 4. Add Regional Metadata

In [ ]:
# Add region and alliance metadata
print("🌍 Adding regional classification...")
df_kpi = generator.add_region_metadata(df_kpi)

print(f"\n✅ Regions Added")
print(f"\nRegional Distribution:")
print(df_kpi['region'].value_counts())

## 5. Add Alliance Metadata

In [ ]:
# Add alliance metadata
print("🤝 Adding alliance metadata...")
df_kpi = generator.add_alliance_metadata(df_kpi)

print(f"\n✅ Alliances Added")
print(f"\nAlliance Membership:")
alliances = ['nato', 'brics', 'asean', 'eu', 'sco']
for alliance in alliances:
    count = df_kpi[alliance].sum() if alliance in df_kpi.columns else 0
    print(f"   {alliance.upper()}: {count} countries")

## 6. Create Output Formats

In [ ]:
# Create wide format (one row per country)
print("📝 Creating wide format...")
df_wide = df_kpi.copy()
df_wide.to_csv('data/processed/military_final_wide.csv', index=False)
df_wide.to_excel('data/processed/military_final.xlsx', index=False)
print(f"   ✓ Saved: military_final_wide.csv ({df_wide.shape})")
print(f"   ✓ Saved: military_final.xlsx ({df_wide.shape})")

# Create long format (one row per country-KPI)
print(f"\n📝 Creating long format...")
df_long = generator.create_long_format(df_kpi)
df_long.to_csv('data/processed/military_final_long.csv', index=False)
print(f"   ✓ Saved: military_final_long.csv ({df_long.shape})")

## 7. KPI Validation

In [ ]:
# Validate KPI calculations
print("\n" + "="*50)
print("KPI VALIDATION REPORT")
print("="*50)

print(f"\n✅ KPI Statistics:")
print(df_wide[['power_index_rank_gap', 'assets_per_capita', 'budget_to_gdp', 
               'personnel_density', 'equipment_density']].describe().round(3))

print(f"\n✅ Data Completeness:")
print(f"   Countries: {len(df_wide)}")
print(f"   Columns: {len(df_wide.columns)}")
print(f"   Missing: {df_wide.isnull().sum().sum()}")
print(f"   Duplicates: {df_wide.duplicated(subset=['country']).sum()}")

print(f"\n✅ Output Files:")
print(f"   • military_final.xlsx ({df_wide.shape[0]} countries, {df_wide.shape[1]} fields)")
print(f"   • military_final_wide.csv (analysis format)")
print(f"   • military_final_long.csv (Tableau format)")
print(f"\n✅ Ready for dashboard development!")
print("="*50)